## Перевод "сырых" данных в маленькие таблицы. Разметки POS-тэгов для человеческих данных

In [2]:
import pandas as pd

In [ ]:
human_data = pd.read_csv('/Users/andrejpihtin/учеба/комп_мага/R/project/raw_data/human_answers.csv')

In [18]:
contexts = pd.unique(human_data['left_context'])

In [19]:
len(contexts)

144

In [21]:
import nltk

# Загружаем необходимые компоненты
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger_ru')


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/andrejpihtin/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_ru to
[nltk_data]     /Users/andrejpihtin/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_ru.zip.


True

In [23]:
text = ", Привет, мир! Python — отличный язык."
tokens = nltk.word_tokenize(text, language="russian")

# Выполняем тегирование для русского языка
pos_tags = nltk.pos_tag(tokens, lang='rus')
print(pos_tags)

[(',', 'NONLEX'), ('Привет', 'S'), (',', 'NONLEX'), ('мир', 'S'), ('!', 'NONLEX'), ('Python', 'NONLEX'), ('—', 'NONLEX'), ('отличный', 'A=m'), ('язык', 'S'), ('.', 'NONLEX')]


In [24]:
human_data.head()

,left_context,target_word,answer,lemma_answer,probability_y
0,У моего отца был счёт в швейцарском,банке,к,к,0.047619
1,У моего отца был счёт в швейцарском,банке,банке,банк,0.952381
2,Во избежание ожогов надо нанести на лицо небол...,количество,количество,количество,1.000000
3,Причиной аварии был мобильный,телефон,звонок,звонок,0.031250
4,Причиной аварии был мобильный,телефон,телефон,телефон,0.968750


In [26]:
import string

# Создаём функцию проверки — есть ли хоть один знак препинания
def has_punctuation(text: str) -> bool:
    return any(ch in string.punctuation for ch in str(text))

test = human_data['answer'].apply(has_punctuation)

In [34]:
human_data[test]

,left_context,target_word,answer,lemma_answer,probability_y
212,Если я еще увижу здесь хоть,соринку,кого-то,кто-то,0.011364
217,Если я еще увижу здесь хоть,соринку,что-то,что-то,0.034091
263,На болотах оставался ещё,лед,какой-то,какой-то,0.007692
577,"Зачем ему звонить, если откликается",спокойный,кто-то,кто-то,0.058824
843,"Существует легенда, что",ноев,кто-то,кто-то,0.017241
860,"Существует легенда, что",ноев,когда-то,когда-то,0.034483
861,"Существует легенда, что",ноев,давным-давно,NOT_FOUND,0.034483
1592,Я сделала,шаг,кое-что,кое-что,0.011364
2139,Душа требовала,красоты,чего-то,что-то,0.184211
2141,Душа требовала,красоты,чего-нибудь,что-нибудь,0.013158


In [ ]:
gpt_data = pd.read_csv('/Users/andrejpihtin/учеба/комп_мага/R/project/raw_data/gpt4omini_morph_2.csv')

In [ ]:
import stanza

nlp = stanza.Pipeline('ru', processors='tokenize,pos')

2026-06-22 03:23:40 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-06-22 03:23:41 INFO: Downloaded file to /Users/andrejpihtin/stanza_resources/resources.json
2026-06-22 03:23:42 INFO: Loading these models for language: ru (Russian):
| Processor | Package          |
--------------------------------
| tokenize  | syntagrus        |
| pos       | syntagrus_charlm |

2026-06-22 03:23:42 INFO: Using device: cpu
2026-06-22 03:23:42 INFO: Loading: tokenize
2026-06-22 03:23:42 INFO: Loading: pos
2026-06-22 03:23:43 INFO: Done loading processors!


In [ ]:
# Создаём функцию, возвращающую список (word, POS) для строки
def get_pos_tag(text: str):
    if not isinstance(text, str) or not text.strip():
        return []  # или return [('','')] / None в зависимости от логики
    doc = nlp(text)
    for sent in doc.sentences:
        for word in sent.words:
            if word.upos == 'PUNCT': 
                continue  # пропускаем пунктуацию
            else: 
                return word.upos # мы ищем только первое слово

In [55]:
get_pos_tag(', привет что-то')

'VERB'

In [57]:
# Пример: применить на столбце answer
human_data['answer_pos'] = human_data['answer'].apply(get_pos_tag)

# Если нужно — можно вывести первые строки
human_data[['answer', 'answer_pos']].head(10)

,answer,answer_pos
0,к,ADP
1,банке,NOUN
2,количество,NOUN
3,звонок,NOUN
4,телефон,NOUN
5,ы,None
6,салфетку,NOUN
7,настойку,NOUN
8,мазь,NOUN
9,строчки,NOUN


In [59]:
human_data[['left_context', 'answer', 'answer_pos']]

,left_context,answer,answer_pos
0,У моего отца был счёт в швейцарском,к,ADP
1,У моего отца был счёт в швейцарском,банке,NOUN
2,Во избежание ожогов надо нанести на лицо небол...,количество,NOUN
3,Причиной аварии был мобильный,звонок,NOUN
4,Причиной аварии был мобильный,телефон,NOUN
...,...,...,...
2654,Когда она в самолёте,не,PART
2655,Когда она в самолёте,у,ADP
2656,Когда она в самолёте,пассажиры,NOUN
2657,Когда она в самолёте,уселась,VERB


In [61]:
gpt_data[['text_test', 'prediction_cleaned', 'upos_word']]

,text_test,prediction_cleaned,upos_word
0,В тот момент,",",PUNCT
1,В тот момент,я,PRON
2,В тот момент,он,PRON
3,В тот момент,у,ADP
4,В тот момент,мне,PRON
...,...,...,...
214087,"Выходя замуж, ты надеялась обрести спокойствие...",быти́йный,ADJ
214088,"Выходя замуж, ты надеялась обрести спокойствие...",быти́и́,NOUN
214089,"Выходя замуж, ты надеялась обрести спокойствие...",быти́нк,NOUN
214090,"Выходя замуж, ты надеялась обрести спокойствие...",быти́нный,ADJ


In [72]:
contexts_intersect = set(pd.unique(gpt_data['text_test'])) & set(pd.unique(human_data['left_context']))

In [73]:
contexts_intersect

{'А промывать манную',
 'Автор принадлежит к числу последних свидетелей',
 'В багажнике были лопата,',
 'В бассейне',
 'В вопросе послышался упрёк',
 'В деревнях по-прежнему',
 'В каждом',
 'В качестве примера приводится',
 'В конверте вместе с деньгами была',
 'В котёл бросают куски',
 'В лесу ветром',
 'В мои обязанности входило утром включить',
 'В речи учёного прозвучало',
 'В современном',
 'В сюжете этого фильма какие-то',
 'В темноте Иван задел острый',
 'В тот момент',
 'В числе возможных кандидатов',
 'Ваня раскрыл было',
 'Взяв с собой фотоаппарат, вся',
 'Власть судов была такой',
 'Во избежание ожогов надо нанести на лицо небольшое',
 'Возможности этих перемен будут обсуждаться в Париже',
 'Володя каким-то образом',
 'Врач прописал заживляющую',
 'Вспоминая',
 'Выходя замуж, ты надеялась обрести спокойствие, уютный',
 'Город, раскинувшийся вдоль',
 'Государством предлагается',
 'Дорога ведет в глухой',
 'Дрозды и сковрцы начали',
 'Думаю, большой',
 'Душа требовала',
 'Ее с

In [77]:
human_data = human_data[human_data['left_context'].isin(contexts_intersect)]

In [78]:
gpt_data = gpt_data[gpt_data['text_test'].isin(contexts_intersect)]

In [79]:
human = human_data[['left_context', 'answer', 'answer_pos']]

In [120]:
gpt = gpt_data[['text_test', 'prediction_cleaned', 'upos_word', 'sum_logprobs']]

In [121]:
gpt = gpt.rename(columns={'text_test': 'context', 'prediction_cleaned': 'word', 'upos_word': 'POS'})

In [105]:
pd.unique(gpt['POS'])

array(['PUNCT', 'PRON', 'ADP', 'SCONJ', 'ADV', 'PART', 'ADJ', 'VERB',
       'NOUN', 'INTJ', 'CCONJ', nan, 'PROPN', 'NUM', 'DET', 'X'],
      dtype=object)

In [86]:
human = human.rename(columns={'left_context': 'context', 'answer': 'word', 'answer_pos': 'POS'})

In [90]:
human = human[~human['POS'].isin(['PUNCT', 'X'])]

In [122]:
human = human.dropna()
gpt = gpt.dropna()

In [97]:
human

,context,word,POS
0,У моего отца был счёт в швейцарском,к,ADP
1,У моего отца был счёт в швейцарском,банке,NOUN
2,Во избежание ожогов надо нанести на лицо небол...,количество,NOUN
3,Причиной аварии был мобильный,звонок,NOUN
4,Причиной аварии был мобильный,телефон,NOUN
...,...,...,...
2654,Когда она в самолёте,не,PART
2655,Когда она в самолёте,у,ADP
2656,Когда она в самолёте,пассажиры,NOUN
2657,Когда она в самолёте,уселась,VERB


In [123]:
gpt = gpt[~gpt['POS'].isin(['PUNCT', 'X'])]

In [101]:
len(human)

2619

In [124]:
# Пример, если в gpt_data есть столбец 'probability'
gpt = (
    gpt
    .sort_values(['context', 'sum_logprobs'], ascending=[True, False])
    .groupby('context')
    .head(15)
    .reset_index(drop=True)
)

In [125]:
pd.unique(gpt['context']).shape

(142,)

In [127]:
gpt.to_csv('/Users/andrejpihtin/учеба/комп_мага/R/project/preprocessed_data/gpt.csv')

In [128]:
human.to_csv('/Users/andrejpihtin/учеба/комп_мага/R/project/preprocessed_data/human.csv')

In [117]:
human['context'].value_counts()

context
Тому, кто                                             63
Я сделала                                             56
Ей никак не суметь                                    56
В бассейне                                            56
На болотах оставался ещё                              53
                                                      ..
Этот роман захватывает читателя с первой               3
Врач прописал заживляющую                              3
Причиной аварии был мобильный                          2
У моего отца был счёт в швейцарском                    2
Во избежание ожогов надо нанести на лицо небольшое     1
Name: count, Length: 142, dtype: int64

In [126]:
gpt['context'].value_counts()

context
А промывать манную                        15
Он ловко поддел концом                    15
По воскресеньям музыканты, исполнявшие    15
Перед ним снова была                      15
Педагог предъявляет                       15
                                          ..
У Пашки                                    5
Я сказал, что русский солдат               4
Шею Лизы украшало ожерелье                 3
Старый шкаф                                3
В числе возможных кандидатов               2
Name: count, Length: 142, dtype: int64

## Перевод POS тэгов в вероятности для каждого контекста

In [129]:
# Для каждого контекста — распределение POS-тегов (в абсолютных числах)
human_pos_by_context = (
    human
    .groupby('context')['POS']
    .value_counts()
    .reset_index(name='count')
)


In [130]:
human_pos_by_context

,context,POS,count
0,А промывать манную,NOUN,2
1,А промывать манную,ADJ,1
2,Автор принадлежит к числу последних свидетелей,NOUN,17
3,Автор принадлежит к числу последних свидетелей,ADJ,6
4,Автор принадлежит к числу последних свидетелей,PRON,3
...,...,...,...
561,"Я слезал, щупал",NOUN,9
562,"Я слезал, щупал",CCONJ,2
563,"Я слезал, щупал",ADP,1
564,"Я слезал, щупал",ADV,1


In [133]:
pos_counts = human.groupby(['context', 'POS']).size().reset_index(name='count')

pos_wide = pos_counts.pivot(index='context', columns='POS', values='count').fillna(0)

human_pos_probs_wide = pos_wide.div(pos_wide.sum(axis=1), axis=0)

# Пример: смотрим первые 2 контекста и все POS-столбцы
print(human_pos_probs_wide.head(2))

POS                                                  ADJ  ADP  ADV  CCONJ  \
context                                                                     
А промывать манную                              0.333333  0.0  0.0    0.0   
Автор принадлежит к числу последних свидетелей  0.193548  0.0  0.0    0.0   

POS                                                  DET  INTJ      NOUN  NUM  \
context                                                                         
А промывать манную                              0.000000   0.0  0.666667  0.0   
Автор принадлежит к числу последних свидетелей  0.064516   0.0  0.548387  0.0   

POS                                             PART      PRON  SCONJ  \
context                                                                 
А промывать манную                               0.0  0.000000    0.0   
Автор принадлежит к числу последних свидетелей   0.0  0.096774    0.0   

POS                                                 VERB  
context       

In [134]:
human_pos_probs_wide

POS,ADJ,ADP,ADV,CCONJ,DET,INTJ,NOUN,NUM,PART,PRON,SCONJ,VERB
context,,,,,,,,,,,,
А промывать манную,0.333333,0.000000,0.000000,0.000000,0.000000,0.0,0.666667,0.000000,0.000000,0.000000,0.000000,0.000000
Автор принадлежит к числу последних свидетелей,0.193548,0.000000,0.000000,0.000000,0.064516,0.0,0.548387,0.000000,0.000000,0.096774,0.000000,0.096774
"В багажнике были лопата,",0.045455,0.000000,0.000000,0.045455,0.000000,0.0,0.909091,0.000000,0.000000,0.000000,0.000000,0.000000
В бассейне,0.107143,0.000000,0.000000,0.000000,0.000000,0.0,0.892857,0.000000,0.000000,0.000000,0.000000,0.000000
В вопросе послышался упрёк,0.125000,0.250000,0.125000,0.250000,0.000000,0.0,0.000000,0.000000,0.000000,0.125000,0.000000,0.125000
...,...,...,...,...,...,...,...,...,...,...,...,...
Я знал: их особенно,0.250000,0.083333,0.083333,0.000000,0.000000,0.0,0.000000,0.000000,0.083333,0.000000,0.000000,0.500000
"Я люблю салат из картошки с зеленью, заправленный",0.666667,0.000000,0.000000,0.000000,0.000000,0.0,0.333333,0.000000,0.000000,0.000000,0.000000,0.000000
Я сделала,0.196429,0.000000,0.053571,0.000000,0.017857,0.0,0.517857,0.053571,0.000000,0.125000,0.017857,0.017857


In [136]:
pos_counts = gpt.groupby(['context', 'POS']).size().reset_index(name='count')

pos_wide = pos_counts.pivot(index='context', columns='POS', values='count').fillna(0)

gpt_pos_probs_wide = pos_wide.div(pos_wide.sum(axis=1), axis=0)

In [137]:
gpt_pos_probs_wide

POS,ADJ,ADP,ADV,CCONJ,DET,INTJ,NOUN,NUM,PART,PRON,PROPN,SCONJ,VERB
context,,,,,,,,,,,,,
А промывать манную,0.066667,0.066667,0.000000,0.000000,0.000000,0.000000,0.800000,0.000000,0.000000,0.000000,0.000000,0.000000,0.066667
Автор принадлежит к числу последних свидетелей,0.333333,0.000000,0.000000,0.111111,0.111111,0.000000,0.333333,0.000000,0.000000,0.111111,0.000000,0.000000,0.000000
"В багажнике были лопата,",0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
В бассейне,0.000000,0.266667,0.000000,0.066667,0.000000,0.200000,0.000000,0.400000,0.000000,0.066667,0.000000,0.000000,0.000000
В вопросе послышался упрёк,0.000000,0.000000,0.133333,0.066667,0.000000,0.066667,0.200000,0.000000,0.066667,0.266667,0.000000,0.066667,0.133333
...,...,...,...,...,...,...,...,...,...,...,...,...,...
Я знал: их особенно,0.066667,0.000000,0.066667,0.000000,0.000000,0.000000,0.000000,0.000000,0.066667,0.000000,0.000000,0.000000,0.800000
"Я люблю салат из картошки с зеленью, заправленный",0.333333,0.066667,0.000000,0.000000,0.000000,0.000000,0.533333,0.000000,0.000000,0.000000,0.000000,0.000000,0.066667
Я сделала,0.000000,0.090909,0.000000,0.000000,0.000000,0.000000,0.000000,0.636364,0.000000,0.181818,0.090909,0.000000,0.000000


In [138]:
human_pos_probs_wide.to_csv('/Users/andrejpihtin/учеба/комп_мага/R/project/preprocessed_data/human_pos_probs.csv')
gpt_pos_probs_wide.to_csv('/Users/andrejpihtin/учеба/комп_мага/R/project/preprocessed_data/gpt_pos_probs.csv')